# EDA: what is associated with Netflix TV series popularity

Cleaning is in `src/tmdb_tv_shows/clean.py`, the reasons are in `tmdb.ipynb`. This notebook only explores. **Assumed** means not verified.

`netflix` = shows where Netflix is in `networks` **or** the `homepage` is on netflix.com (2,031 shows).

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tmdb_tv_shows.clean import build

raw, clean, dedup, dropped = build()
netflix = dedup[dedup.is_netflix].copy()
print(len(dedup), len(netflix))   # 164705 2031

164705 2031


## 1. Data quality

Checks on the 2,031 Netflix shows: missing data, flags and zeros, columns, Netflix coverage, outliers, then a summary.

### Missing data

- `tagline` 80.9% missing. A marketing slogan, **assumed** weak from domain knowledge. Not used.
- `created_by` 45.6% missing. Interesting, but imputing creators is not feasible. No imputation.
- Usable with a missing-value approach: `episode_run_time` 28.6%, `production_companies` 27.4%, `production_countries` 12.3%, `vote_average` 12.1% (no votes).
- `networks` is 2.0% missing, the homepage-only shows with a blank `networks`.
- The table below also lists our added columns, so some rows repeat (`genres` and `genres_list`).

In [11]:
cols = list(netflix.columns)
missing = netflix[cols].isna().agg(["sum", "mean"]).T
missing["mean"] = (missing["mean"] * 100).round(1)
missing.sort_values("mean", ascending=False)

,sum,mean
tagline,1643.0,80.9
created_by,926.0,45.6
created_by_list,926.0,45.6
episode_run_time,580.0,28.6
production_companies_list,556.0,27.4
production_companies,556.0,27.4
production_countries,250.0,12.3
vote_average,246.0,12.1
last_air_date,210.0,10.3
homepage,186.0,9.2


### Flags and zeros

- `dates_reversed` and `votes_inconsistent` are 0 in Netflix.
- 246 shows have 0 votes, so `vote_average` is missing. 548 have 5 or fewer votes.
- `popularity` is 0 for 4 shows, the smallest positive value is 0.6.
- 153 shows have no `first_air_date` and 4 start after 2024-04-04 (157 not launched by date).

In [12]:
print(netflix.dates_reversed.sum(), netflix.votes_inconsistent.sum())               # 0 0
print((netflix.vote_count == 0).sum(), (netflix.vote_count <= 5).sum())            # 246 548
print((netflix.popularity == 0).sum(), netflix.popularity[netflix.popularity > 0].min())   # 4 0.6
print(netflix.first_air_date.isna().sum(), (netflix.first_air_date > "2024-04-04").sum())  # 153 4

0 0
246 548
4 0.6
153 4


### Columns with little information or overlap

- `in_production` repeats `status` exactly.
- `adult` is True for 3 shows, almost constant.
- `type` has 5 values (Scripted, Miniseries, Reality, Documentary, Talk Show), usable. **Assumed** known before launch.
- `original_language` equals the first of `languages` in about 91% of shows, so they overlap.

In [13]:
orig = list(raw.columns)
print(netflix[orig].astype(str).nunique().sort_values().head(8))
print(netflix.adult.value_counts())   # False 2028, True 3
print(netflix.type.value_counts())
print(pd.crosstab(netflix.status, netflix.in_production))
print((netflix.original_language == netflix.languages.str.split(", ").str[0]).mean())

in_production         2
adult                 2
status                5
type                  5
number_of_seasons    13
original_language    34
origin_country       81
episode_run_time     90
dtype: int64
adult
False    2028
True        3
Name: count, dtype: int64
type
Scripted       1165
Miniseries      447
Reality         212
Documentary     185
Talk Show        22
Name: count, dtype: int64
in_production     False  True 
status                        
Canceled            213      0
Ended              1009      0
In Production         0    151
Planned               0     42
Returning Series      0    616
0.913343180699163


### Netflix coverage

- TMDB's guideline says `networks` typically lists only the first network to air a show, so licensed shows (Friends on NBC) list their original network.
- 1,929 shows match by network, 1,718 by netflix.com homepage, 1,616 both. 102 are homepage only: 41 with blank `networks`, 61 with other networks.
- Among the network matches, 181 list other networks too (YouTube, Channel 3 and others).
- `networks` is never changed. The homepage match is its own column.

In [14]:
print(netflix.netflix_network.sum(), netflix.netflix_homepage.sum(),
      (netflix.netflix_network & netflix.netflix_homepage).sum())     # 1929 1718 1616
h = netflix[~netflix.netflix_network & netflix.netflix_homepage]
print(len(h), h.networks.isna().sum(), h.networks.notna().sum())      # 102 41 61
nn = netflix[netflix.netflix_network]
k = nn.networks_list.apply(len)
print((k == 1).sum(), (k > 1).sum())                                  # 1748 181
h[h.networks.notna()][["name", "networks"]].head()

1929 1718 1616
102 41 61
1748 181


,name,networks
27,Friends,NBC
3725,Top Boy,Channel 4
6894,Rabbids Invasion,"France 3, Nicktoons"
12009,Some Assembly Required,YTV
5379,Historia de un clan,Telefe


### Outliers

- Seasons (max 30) and episodes (max 973) are long-running shows, genuine. Power Rangers, Pen Tor (817), Bakugan, Friends.
- `episode_run_time` max 515 (Tunnel, 16 episodes). The largest look like total runtimes, not per episode. **Assumed**, not proven. None above 600.
- Votes and popularity are very skewed but real (Money Heist 17,836 votes, Sex Education popularity 1,009).
- Kept everything. Use log scales or medians in EDA.

In [15]:
num = ["number_of_seasons", "number_of_episodes", "episode_run_time", "vote_count", "popularity"]
print(netflix[num].describe(percentiles=[.5, .9, .99]).round(1).T)
big = netflix.nlargest(5, "episode_run_time")[["name", "number_of_episodes", "episode_run_time"]]
big.assign(per_episode=big.episode_run_time / big.number_of_episodes)

                     count   mean    std  min   50%    90%     99%      max
number_of_seasons   2031.0    1.6    1.5  1.0   1.0    3.0     7.0     30.0
number_of_episodes  2029.0   15.2   34.2  0.0   8.0   32.0    92.4    973.0
episode_run_time    1451.0   41.0   25.4  3.0  41.0   60.0   107.5    515.0
vote_count          2031.0  191.7  891.0  0.0  20.0  313.0  2799.2  17836.0
popularity          2031.0   15.6   40.8  0.0   6.5   31.7   144.1   1009.0


,name,number_of_episodes,episode_run_time,per_episode
41246,Tunnel,16.0,515.0,32.1875
2687,The One,8.0,337.0,42.1250
5184,The Innocent Man,6.0,288.0,48.0000
36408,Sunshine Eyes,10.0,275.0,27.5000
5569,The Pharmacist,4.0,217.0,54.2500


### Data quality summary

- **Complete or nearly:** `id`, `name`, `type`, `status`, `popularity`, `vote_count`, `original_language`, `number_of_seasons`, plus `genres`, `origin_country`, `languages`, `overview` (each under 4% missing).
- **Usable with a missing-value approach:** `episode_run_time`, `production_companies`, `production_countries`, `vote_average`.
- **Weak or not used:** `tagline`, `created_by`, `in_production`, `adult`, links and images.
- **Cautions:** popularity is one noisy snapshot, `vote_average` is unreliable for few votes, some runtimes may be totals, the Netflix label depends on `networks` plus homepage, the snapshot date (2024-04-04) is inferred, 157 shows are not launched by date.

In [16]:
SNAPSHOT = pd.Timestamp("2024-04-04")   # latest last_air_date in the data, assumed capture date

launched = netflix[netflix.first_air_date.notna() & (netflix.first_air_date <= SNAPSHOT)].copy()
print(len(netflix), len(launched))      # expect 2031 1874
launched.status.value_counts()


2031 1874


status
Ended               1009
Returning Series     608
Canceled             212
In Production         44
Planned                1
Name: count, dtype: int64